## Imports

Libraries

In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import imageio.v2 as imageio
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

Paths

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.paths import SOCAT_FILE

SOCAT_LOAD_PATH = SOCAT_FILE

Settings

In [ ]:
SOCAT_START = "1993-01-01"
SOCAT_END = "2024-12-31"

## Load SOCAT

In [ ]:
ds = xr.open_dataset(SOCAT_LOAD_PATH)
ds = ds.sel(tmnth=slice(SOCAT_START, SOCAT_END))

print(ds)

/home/efs25/miniconda3/envs/irp-efs25/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


<xarray.Dataset> Size: 3GB
Dimensions:                (tmnth: 384, bnds: 2, ylat: 180, xlon: 360)
Coordinates:
  * tmnth                  (tmnth) datetime64[ns] 3kB 1993-01-16T12:00:00 ......
  * ylat                   (ylat) float64 1kB -89.5 -88.5 -87.5 ... 88.5 89.5
  * xlon                   (xlon) float64 3kB -179.5 -178.5 ... 178.5 179.5
Dimensions without coordinates: bnds
Data variables: (12/25)
    tmnth_bnds             (tmnth, bnds) datetime64[ns] 6kB ...
    count_ncruise          (tmnth, ylat, xlon) float64 199MB ...
    fco2_count_nobs        (tmnth, ylat, xlon) float64 199MB ...
    fco2_ave_weighted      (tmnth, ylat, xlon) float32 100MB ...
    fco2_ave_unwtd         (tmnth, ylat, xlon) float32 100MB ...
    fco2_min_unwtd         (tmnth, ylat, xlon) float32 100MB ...
    ...                     ...
    salinity_min_unwtd     (tmnth, ylat, xlon) float32 100MB ...
    salinity_max_unwtd     (tmnth, ylat, xlon) float32 100MB ...
    salinity_std_weighted  (tmnth, ylat, x

### Create monthly SOCAT observation GIF

In [5]:
output_path = os.path.join(
    ROOT,
    "figures",
    "socat_monthly_observations.gif",
)

# Number of fCO2 measurements per grid cell per month
count_var = "fco2_count_nobs"

# Fixed colour scale across ALL months so that the animation
# shows genuine changes in sampling coverage.
all_counts = ds[count_var].where(ds[count_var] > 0)

# Use a fixed upper limit so that a few heavily sampled cells
# do not dominate the colour scale.
vmax = float(all_counts.quantile(0.99))

print(f"99th percentile of monthly observations per grid cell: {vmax:.0f}")

norm = mcolors.LogNorm(
    vmin=1,
    vmax=vmax,
)

# cmap = cmocean.cm.dense.copy()
# cmap.set_bad("white")
cmap = cmocean.cm.dense.copy()
cmap.set_bad((1, 1, 1, 0))  # transparent masked cells


with imageio.get_writer(
    output_path,
    mode="I",
    fps=8,
    loop=0,
    subrectangles=True,
) as writer:

    for t_i, t in enumerate(ds.tmnth.values):

        # Monthly observation count
        field = ds[count_var].isel(tmnth=t_i).where(
            ds[count_var].isel(tmnth=t_i) > 0
        )

        fig = plt.figure(
            figsize=(10, 5.5),
            dpi=80,
        )

        ax = fig.add_axes(
            [0.02, 0.05, 0.88, 0.85],
            projection=ccrs.Robinson(),
        )

        ax.set_global()

        mesh = ax.pcolormesh(
            ds["xlon"],
            ds["ylat"],
            field,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            norm=norm,
            shading="auto",
        )

        ax.add_feature(
            cfeature.LAND,
            facecolor="#dddee1",
            edgecolor="none",
            zorder=3,
        )

        ax.coastlines(
            linewidth=0.5,
            zorder=4,
        )

        ax.add_feature(
            cfeature.BORDERS,
            linewidth=0.3,
            zorder=4,
        )

        # Colourbar with triangular extensions
        cax = fig.add_axes(
            [0.91, 0.25, 0.015, 0.5]
        )

        cbar = fig.colorbar(
            mesh,
            cax=cax,
            orientation="vertical",
            extend="both",
        )

        cbar.set_label(
            "Number of fCO₂ observations",
            fontsize=12,
        )

        # Month/year
        date = np.datetime_as_string(
            t,
            unit="M",
        )

        fig.suptitle(
            rf"SOCAT$^2$ observations - {date}",
            fontsize=18,
            fontweight="bold",
            y=0.94,
        )

        fig.canvas.draw()

        frame = np.asarray(
            fig.canvas.buffer_rgba()
        )[:, :, :3]

        writer.append_data(frame)

        plt.close(fig)

print(f"Saved GIF to:\n{output_path}")

99th percentile of monthly observations per grid cell: 1291
Saved GIF to:
/home/efs25/GitHub/irp-efs25/figures/socat_monthly_observations.gif


In [4]:
# output_path = os.path.join(
#     ROOT,
#     "figures",
#     "socat_monthly_observations.gif",
# )

# # Use the number of observations in each grid cell
# # This shows the spatial sparsity/unevenness directly
# count_var = "fco2_ave_weighted"

# # Fixed colour scale across ALL months so that changes in
# # coverage are meaningful rather than caused by rescaling
# all_counts = ds[count_var].where(ds[count_var] > 0)

# vmax = float(all_counts.max())
# print(f"Maximum monthly observations in one grid cell: {vmax:.0f}")

# norm = mcolors.LogNorm(vmin=1, vmax=vmax)

# cmap = cmocean.cm.matter.copy()
# cmap.set_bad("white")


# with imageio.get_writer(
#     output_path,
#     mode="I",
#     fps=8,
#     loop=0,
#     subrectangles=True,
# ) as writer:

#     for t_i, t in enumerate(ds.tmnth.values):

#         # Monthly observations
#         field = ds[count_var].isel(tmnth=t_i).where(
#             ds[count_var].isel(tmnth=t_i) > 0
#         )

#         fig = plt.figure(figsize=(10, 5.5), dpi=80)

#         ax = fig.add_axes(
#             [0.02, 0.05, 0.88, 0.85],
#             projection=ccrs.Robinson(),
#         )

#         ax.set_global()

#         mesh = ax.pcolormesh(
#             ds["xlon"],
#             ds["ylat"],
#             field,
#             transform=ccrs.PlateCarree(),
#             cmap=cmap,
#             norm=norm,
#             shading="auto",
#         )

#         ax.add_feature(
#             cfeature.LAND,
#             facecolor="#dddee1",
#             edgecolor="none",
#             zorder=3,
#         )

#         ax.coastlines(
#             linewidth=0.5,
#             zorder=4,
#         )

#         ax.add_feature(
#             cfeature.BORDERS,
#             linewidth=0.3,
#             zorder=4,
#         )

#         # Colourbar
#         cax = fig.add_axes([0.91, 0.25, 0.025, 0.5])

#         cbar = fig.colorbar(
#             mesh,
#             cax=cax,
#             orientation="vertical",
#         )

#         cbar.set_label(
#             "SOCAT observations",
#             fontsize=9,
#         )

#         # Month/year
#         date = np.datetime_as_string(t, unit="M")

#         fig.suptitle(
#             f"SOCAT observations — {date}",
#             fontsize=13,
#             fontweight="bold",
#             y=0.94,
#         )

#         fig.canvas.draw()

#         frame = np.asarray(
#             fig.canvas.buffer_rgba()
#         )[:, :, :3]

#         writer.append_data(frame)

#         plt.close(fig)

# print(f"Saved GIF to:\n{output_path}")